# Training Trajectories Analysis - Sequential Groups

Questo notebook analizza le traiettorie dei pedoni durante il training, mostrando:
- Gruppi di 10 pedoni alla volta
- Traiettoria completa di ogni pedone dall'inizio alla fine
- Analisi sequenziale dell'evoluzione del training

##  Import Libraries

In [ ]:
%matplotlib inline

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from typing import Dict, List, Tuple
import seaborn as sns


plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
mpl.rcParams['figure.figsize'] = (12, 8)
mpl.rcParams['figure.dpi'] = 100

##  Configuration

In [ ]:
# Path base delle session di training
BASE_PATH = "../output/runs/training/"

# Nome della session da analizzare (lascia None per selezionare la più recente)
SESSION_NAME = None  # es: "session_2025-10-07_23-52-13" oppure None

# Numero di istanze parallele (di solito 10)
NUM_INSTANCES = 10

# Numero di pedoni per grafico
PEDESTRIANS_PER_PLOT = 10

# Framerate (fps) - da constants.gd: PHYSICS_TICKS_PER_SECONDS / TICKS_BETWEEN_LOG
FRAMERATE = 30  # 60 / 2 = 30 fps

# Colori per i pedoni in ogni grafico
PEDESTRIAN_COLORS = plt.cm.tab10(np.linspace(0, 1, 10))


## Session Discovery

In [ ]:
def find_sessions(base_path: str) -> List[str]:
    """Trova tutte le session di training disponibili."""
    if not os.path.exists(base_path):
        print(f" Path non trovato: {base_path}")
        return []

    sessions = []
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path) and item.startswith('session_'):
            sessions.append(item)

    return sorted(sessions, reverse=True)

def find_levels(session_path: str) -> List[str]:
    """Trova tutti i livelli in una session."""
    levels = []
    for item in os.listdir(session_path):
        item_path = os.path.join(session_path, item)
        if os.path.isdir(item_path) and item.endswith('Batch'):
            level_name = item[:-5]
            levels.append(level_name)
    return sorted(levels)

# Trova sessions disponibili
available_sessions = find_sessions(BASE_PATH)

if not available_sessions:
    print(" Nessuna session trovata!")
else:
    print(f"\n Session disponibili ({len(available_sessions)}):")
    for i, sess in enumerate(available_sessions, 1):
        print(f"  {i}. {sess}")

    if SESSION_NAME is None:
        selected_session = available_sessions[0]
        print(f"\n Session selezionata automaticamente (più recente): {selected_session}")
    else:
        selected_session = SESSION_NAME
        print(f"\n Session selezionata manualmente: {selected_session}")

    SESSION_PATH = os.path.join(BASE_PATH, selected_session)

    available_levels = find_levels(SESSION_PATH)
    print(f"\n Livelli trovati ({len(available_levels)}):")
    for i, level in enumerate(available_levels, 1):
        print(f"  {i}. {level}")

##  Data Loading Functions

In [ ]:
def load_trajectories(level_name: str, session_path: str) -> pd.DataFrame:
    """
    Carica le traiettorie di un livello.
    """
    file_path = os.path.join(session_path, f"{level_name}Batch", "trajectories.txt")

    if not os.path.exists(file_path):
        print(f" File non trovato: {file_path}")
        return None

    df = pd.read_csv(
        file_path,
        sep=r'\s+',
        comment='#',
        header=None,
        names=['id', 'frame', 'x', 'y', 'z', 'combined_group']
    )

    df['instance_id'] = df['combined_group'] // 10000
    df['collision_group'] = df['combined_group'] % 10000

    print(f" Caricato {level_name}: {len(df)} righe, {df['frame'].max()+1} frame")
    print(f"   Istanze trovate: {sorted(df['instance_id'].unique())}")
    print(f"   Pedoni unici: {df['id'].nunique()}")

    return df

def get_pedestrian_groups(df: pd.DataFrame, peds_per_group: int = 10) -> List[List]:
    """
    Divide tutti i pedoni in gruppi di dimensione fissa.
    I pedoni sono ordinati cronologicamente (per primo frame di apparizione).
    """
    # Ottieni primo frame di apparizione per ogni pedone
    first_frame = df.groupby('id')['frame'].min().sort_values()
    all_pedestrian_ids = first_frame.index.tolist()

    # Dividi in gruppi
    groups = []
    for i in range(0, len(all_pedestrian_ids), peds_per_group):
        group = all_pedestrian_ids[i:i+peds_per_group]
        groups.append(group)

    print(f"\n Creati {len(groups)} gruppi di pedoni:")
    for i, group in enumerate(groups):
        print(f"   Gruppo {i+1}: {len(group)} pedoni")

    return groups

##  Plotting Functions

In [ ]:
def plot_pedestrian_group(df: pd.DataFrame, pedestrian_ids: List,
                          level_name: str, group_num: int, total_groups: int):
    """
    Plot delle traiettorie complete di un gruppo di pedoni.
    """
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))

    # Titolo con informazioni sul gruppo
    fig.suptitle(f'{level_name} - Gruppo {group_num}/{total_groups} '
                 f'(Pedoni {group_num*PEDESTRIANS_PER_PLOT - PEDESTRIANS_PER_PLOT + 1}-'
                 f'{group_num*PEDESTRIANS_PER_PLOT - PEDESTRIANS_PER_PLOT + len(pedestrian_ids)})',
                 fontsize=16, fontweight='bold')

    # Plot per ogni pedone nel gruppo
    for idx, ped_id in enumerate(pedestrian_ids):
        ped_df = df[df['id'] == ped_id].sort_values('frame')

        if len(ped_df) == 0:
            continue

        color = PEDESTRIAN_COLORS[idx % 10]

        # Plot traiettoria
        ax.plot(ped_df['x'], ped_df['y'],
               alpha=0.7, linewidth=2, color=color,
               label=f'Ped {idx+1}')

        # Punto di inizio (cerchio)
        start_point = ped_df.iloc[0]
        ax.scatter(start_point['x'], start_point['y'],
                  c=[color], s=150, marker='o',
                  edgecolors='black', linewidths=2, zorder=5)

        # Punto di fine (X)
        end_point = ped_df.iloc[-1]
        ax.scatter(end_point['x'], end_point['y'],
                  c=[color], s=150, marker='X',
                  edgecolors='black', linewidths=2, zorder=5)

        # Aggiungi freccia direzionale a metà traiettoria
        mid_idx = len(ped_df) // 2
        if mid_idx > 0 and mid_idx < len(ped_df) - 1:
            x1, y1 = ped_df.iloc[mid_idx]['x'], ped_df.iloc[mid_idx]['y']
            x2, y2 = ped_df.iloc[mid_idx+1]['x'], ped_df.iloc[mid_idx+1]['y']
            ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                       arrowprops=dict(arrowstyle='->', color=color, lw=2))

    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Y (m)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, ncol=1)

    # Aggiungi informazioni sul grafico
    info_text = f'○ = Start | X = End | → = Direzione'
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
           fontsize=10, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()

def plot_all_groups(df: pd.DataFrame, level_name: str, peds_per_group: int = 10):
    """
    Crea un grafico per ogni gruppo di pedoni.
    """
    groups = get_pedestrian_groups(df, peds_per_group)
    total_groups = len(groups)

    print(f"\n Creazione di {total_groups} grafici...\n")

    for i, group in enumerate(groups, 1):
        print(f" Grafico {i}/{total_groups} - {len(group)} pedoni")
        plot_pedestrian_group(df, group, level_name, i, total_groups)

    print(f"\n Completati tutti i {total_groups} grafici!")

##  Analysis: Visualizza Gruppi Sequenziali

In [ ]:
LEVEL_TO_ANALYZE = "labyrinth"  # Cambia questo con il nome del livello

if LEVEL_TO_ANALYZE not in available_levels:
    print(f" ERRORE: Livello '{LEVEL_TO_ANALYZE}' non trovato!")
    print(f" Livelli disponibili: {available_levels}")
else:
    print(f"\n" + "="*60)
    print(f" ANALISI SEQUENZIALE: {LEVEL_TO_ANALYZE}")
    print("="*60)

    # Carica dati
    df = load_trajectories(LEVEL_TO_ANALYZE, SESSION_PATH)

    if df is not None:
        # Genera tutti i grafici
        plot_all_groups(df, LEVEL_TO_ANALYZE, PEDESTRIANS_PER_PLOT)

##  Statistics: Analisi per Gruppo

In [ ]:
def compute_group_statistics(df: pd.DataFrame, groups: List[List]) -> pd.DataFrame:
    """
    Calcola statistiche per ogni gruppo di pedoni.
    """
    stats = []

    for i, group in enumerate(groups, 1):
        group_df = df[df['id'].isin(group)].copy()

        # Calcola velocità
        group_df = group_df.sort_values(['id', 'frame'])
        group_df['dx'] = group_df.groupby('id')['x'].diff()
        group_df['dy'] = group_df.groupby('id')['y'].diff()
        group_df['distance'] = np.sqrt(group_df['dx']**2 + group_df['dy']**2)
        group_df['speed'] = group_df['distance'] * FRAMERATE

        stats.append({
            'Gruppo': i,
            'Num_Pedoni': len(group),
            'Velocità_Media': group_df['speed'].mean(),
            'Velocità_Max': group_df['speed'].max(),
            'Distanza_Totale': group_df['distance'].sum(),
            'Frame_Totali': group_df['frame'].nunique(),
            'Punti_Dati': len(group_df)
        })

    return pd.DataFrame(stats)

# Calcola statistiche se abbiamo dati
if df is not None:
    groups = get_pedestrian_groups(df, PEDESTRIANS_PER_PLOT)
    stats_df = compute_group_statistics(df, groups)

    print("\n STATISTICHE PER GRUPPO:")
    print(stats_df.to_string(index=False))

    # Plot evoluzione velocità media per gruppo
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    ax.plot(stats_df['Gruppo'], stats_df['Velocità_Media'],
           marker='o', linewidth=2, markersize=8)
    ax.set_xlabel('Numero Gruppo (Ordine Cronologico)', fontsize=12)
    ax.set_ylabel('Velocità Media (m/s)', fontsize=12)
    ax.set_title(f'{LEVEL_TO_ANALYZE} - Evoluzione Velocità Media per Gruppo',
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

##  Export Results

In [ ]:
if df is not None and len(stats_df) > 0:
    output_path = os.path.join(SESSION_PATH, f"{LEVEL_TO_ANALYZE}_group_statistics.csv")
    stats_df.to_csv(output_path, index=False)
    print(f"\n Statistiche esportate in: {output_path}")